# 3-level autoencoder · 2000 epoch results
Portable copy of the completed 20260917 experiment. Original model and training loop retained.
Default: load final weights and reconstruct training samples 100 and 200. Set AE_DATASET_ROOT to the folder containing train/ and test/.
To retrain from scratch, set RUN_TRAINING=True; 2000 epochs, seed 20260917, batch 4, Adam 0.001. New outputs go to runs/fresh_training/.
Select GPUs via CUDA_VISIBLE_DEVICES before launching Jupyter. CPU inference is supported but slow.


In [ ]:
# Run from this folder in a fresh kernel.
import os, sys, random, json, math
from pathlib import Path
# Select a GPU before starting Jupyter, e.g. CUDA_VISIBLE_DEVICES=1 jupyter lab.
# No physical GPU is selected automatically.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
project_dir = Path.cwd().resolve()
if not (project_dir / "source" / "dataloader.py").exists():
    raise RuntimeError("Start Jupyter from the downloaded experiment folder.")
sys.path.insert(0, str(project_dir))
import torch, numpy as np
from torch.utils.data import Dataset, DataLoader
SEED = 20260917
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
# Default: load a published checkpoint; do not retrain or overwrite results.
RUN_TRAINING = False
CHECKPOINT_KIND = "final"  # "best" selects the lowest validation loss.
DATASET_ROOT = Path(os.environ.get("AE_DATASET_ROOT", str(project_dir / "data" / "balls_128"))).expanduser()
experiment_dir = project_dir / "runs" / "fresh_training" if RUN_TRAINING else project_dir
for directory in ("results", "checkpoint"):
    (experiment_dir / directory).mkdir(parents=True, exist_ok=True)
run_name = "train_cnn_vae_3_level"


In [ ]:
from importlib import reload
import source.dataloader as dataloader_module
from source.utility import play_dataset_item

# Reload local edits when this cell is rerun in an existing notebook kernel.
reload(dataloader_module)
BallMovingDataset = dataloader_module.BallMovingDataset

train_path = DATASET_ROOT / "train"
test_path = DATASET_ROOT / "test"
ball_moving_dataset_train = BallMovingDataset(train_path)
ball_moving_dataset_test = BallMovingDataset(test_path)

train_loader = DataLoader(
    ball_moving_dataset_train,
    batch_size=4,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,  # Reliable default inside a notebook.
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    ball_moving_dataset_test,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

actions_batch, frames_batch, instructions_batch, sample_indices = next(iter(train_loader))
print(f"train samples: {len(ball_moving_dataset_train)}")
print(f"test samples:  {len(ball_moving_dataset_test)}")
print(f"frames: {frames_batch.shape}, {frames_batch.dtype}")
print(f"actions: {actions_batch.shape}, {actions_batch.dtype}")
print(f"instructions: {len(instructions_batch)} strings")
print(f"sample indices: {sample_indices.tolist()}")

In [ ]:
import torch.nn as nn


class symmetrical_autoencoder(nn.Module):
    """A 3D autoencoder assembled from repeated, mirrored blocks."""

    def __init__(
        self,
        input_dim,
        channel_numbers=(3, 16, 32, 64),
        depth_scales=(2, 2, 2),
    ):
        super().__init__()

        self._validate_configuration(input_dim, channel_numbers, depth_scales)

        encoder_layers = []

        print(list(zip(
            self.channel_numbers[:-1],
            self.channel_numbers[1:],
            self.depth_scales,
        )))

        # Shortcut: repeat Conv3d -> ReLU -> Conv3d -> ReLU -> MaxPool3d.
        for input_channels, output_channels, scale in zip(
            self.channel_numbers[:-1],
            self.channel_numbers[1:],
            self.depth_scales,
        ):
            encoder_layers.extend(
                [
                    nn.Conv3d(input_channels, output_channels, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv3d(output_channels, output_channels, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.MaxPool3d(kernel_size=scale, stride=scale),
                ]
            )

        self.encoders = nn.Sequential(*encoder_layers)
        self.decoders = self.symmetrical_decoder()

    # Validation
    def _validate_configuration(self, input_dim, channel_numbers, depth_scales):
        """Normalize configuration and check it before constructing layers."""
        self.channel_numbers = list(channel_numbers)
        if len(self.channel_numbers) < 2:
            raise ValueError("channel_numbers must contain at least input and output channels")
        if any(channels < 1 for channels in self.channel_numbers):
            raise ValueError("Every channel number must be positive")

        self.input_dim = (input_dim,) * 3 if isinstance(input_dim, int) else tuple(input_dim)
        if len(self.input_dim) != 3:
            raise ValueError("input_dim must be an int or a (T, H, W) tuple")

        if len(depth_scales) != len(self.channel_numbers) - 1:
            raise ValueError(
                "depth_scales must have one entry per encoder block "
                "(len(channel_numbers) - 1)"
            )
        self.depth_scales = [
            (scale,) * 3 if isinstance(scale, int) else tuple(scale)
            for scale in depth_scales
        ]
        if any(len(scale) != 3 or any(value < 1 for value in scale)
               for scale in self.depth_scales):
            raise ValueError("Each depth scale must be positive and have the form (T, H, W)")

        total_scale = [1, 1, 1]
        for scale in self.depth_scales:
            total_scale = [old * new for old, new in zip(total_scale, scale)]
        if any(size % scale != 0 for size, scale in zip(self.input_dim, total_scale)):
            raise ValueError(
                f"Input dimensions {self.input_dim} must be divisible by {tuple(total_scale)}"
            )

    @staticmethod
    def _validate_mirrored_channels(layer, current_channels):
        if current_channels != layer.out_channels:
            raise ValueError("Encoder channel sequence cannot be mirrored")

    def encode(self, x):
        return self.encoders(x)

    def decode(self, latent):
        return self.decoders(latent)

    def symmetrical_decoder(self):
        """Mirror every encoder layer using explicit construction rules."""

        def mirror_conv3d(layer, current_channels):
            self._validate_mirrored_channels(layer, current_channels)
            mirrored = nn.ConvTranspose3d(
                in_channels=layer.out_channels,
                out_channels=layer.in_channels,
                kernel_size=layer.kernel_size,
                stride=layer.stride,
                padding=layer.padding,
                dilation=layer.dilation,
                groups=layer.groups,
                bias=layer.bias is not None,
            )
            return mirrored, layer.in_channels

        def mirror_maxpool3d(layer, current_channels):
            stride = layer.stride if layer.stride is not None else layer.kernel_size
            mirrored = nn.ConvTranspose3d(
                in_channels=current_channels,
                out_channels=current_channels,
                kernel_size=layer.kernel_size,
                stride=stride,
                padding=layer.padding,
            )
            return mirrored, current_channels

        def mirror_relu(layer, current_channels):
            return nn.ReLU(inplace=layer.inplace), current_channels

        decoder_mapping = {
            nn.Conv3d: mirror_conv3d,
            nn.MaxPool3d: mirror_maxpool3d,
            nn.ReLU: mirror_relu,
        }

        encoder_layers = list(self.encoders)
        current_channels = next(
            layer.out_channels
            for layer in reversed(encoder_layers)
            if isinstance(layer, nn.Conv3d)
        )
        decoder_layers = []

        for encoder_layer in reversed(encoder_layers):
            constructor = decoder_mapping.get(type(encoder_layer))
            if constructor is None:
                raise TypeError(
                    f"No decoder mapping for {type(encoder_layer).__name__}"
                )
            decoder_layer, current_channels = constructor(
                encoder_layer, current_channels
            )
            decoder_layers.append(decoder_layer)

        decoder_layers.append(nn.Sigmoid())
        return nn.Sequential(*decoder_layers)

    def forward(self, x):
        latent = self.encode(x)
        reconstruction = self.decode(latent)
        return reconstruction

autoencoder_1 = symmetrical_autoencoder(
    input_dim=128,
    channel_numbers=[3, 16, 32, 64],
    depth_scales=[2, 2, 2],
)


In [ ]:
if RUN_TRAINING:
    # Place the model on the GPU before creating the optimizer or running inference.
    model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    autoencoder_1 = autoencoder_1.to(model_device)
    print(f"Training device: {model_device}")
    if model_device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(model_device)}")
    
    num_epochs = 2000
    
    actions, frames, instructions, sample_indices = next(iter(ball_moving_dataset_train))
    
    # Input frame formatting: 
    
    # print(actions, frames, instructions, sample_indices)
    print(frames.shape)
    print(type(frames))
    x = frames.unsqueeze(0).permute(0, 4, 1, 2, 3).contiguous().float().to(model_device) / 255.0
    latent = autoencoder_1.encode(x)
    recover_x = autoencoder_1.decode(latent)
    
    print(x.shape)
    print(latent.shape)
    print(recover_x.shape)
    
    # training code
    max_epoch = 2000
    loss_thres = 1e-6
    print(len(train_loader))
    
    optimizer = torch.optim.Adam(autoencoder_1.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    from time import perf_counter
    from functools import partial
    from tqdm.auto import tqdm as _tqdm
    tqdm = partial(_tqdm, disable=os.environ.get("AE_BATCH") == "1")
    from IPython.display import display
    import matplotlib.pyplot as plt
    
    # This history starts fresh each time this cell runs.
    loss_history = {"train": [], "validation": []}
    best_validation_loss = float("inf")
    plot_display = None if os.environ.get("AE_BATCH") == "1" else display("Preparing loss chart...", display_id=True)
    
    
    with tqdm(range(max_epoch), desc="Epochs", unit="epoch") as epoch_progress:
        for i in epoch_progress:
            started = perf_counter()
            training_loss_sum = 0.0
            training_samples = 0
            autoencoder_1.train()
    
            with torch.enable_grad():
                with tqdm(train_loader, desc="Training", unit="batch", leave=False) as batch_progress:
                    for actions, frames, instructions, sample_indices in batch_progress:
                        # Loader batches already have shape [B, T, H, W, C].
                        x = frames.permute(0, 4, 1, 2, 3).contiguous().float().to(model_device) / 255.0
                        latent = autoencoder_1.encode(x)
                        recover_x = autoencoder_1.decode(latent)
                        loss = criterion(x, recover_x)
                        optimizer.zero_grad();
                        loss.backward();
                        optimizer.step();
    
                        training_loss_sum += loss.item() * x.size(0)
                        training_samples += x.size(0)
                        batch_progress.set_postfix(samples=training_samples, loss=f"{loss.item():.6f}", mean=f"{training_loss_sum / training_samples:.6f}")
    
            # Use the existing test split for validation, with the same raw pixel scale.
            validation_loss_sum = 0.0
            validation_samples = 0
            autoencoder_1.eval()
            try:
                with torch.no_grad():
                    with tqdm(test_loader, desc="Validation", unit="batch", leave=False) as validation_progress:
                        for actions, frames, instructions, sample_indices in validation_progress:
                            validation_x = frames.permute(0, 4, 1, 2, 3).contiguous().float().to(model_device) / 255.0
                            validation_reconstruction = autoencoder_1.decode(autoencoder_1.encode(validation_x))
                            validation_loss = criterion(validation_x, validation_reconstruction)
                            validation_loss_sum += validation_loss.item() * validation_x.size(0)
                            validation_samples += validation_x.size(0)
                            validation_progress.set_postfix(mean=f"{validation_loss_sum / validation_samples:.6f}")
            finally:
                autoencoder_1.train()
    
            train_mean = training_loss_sum / training_samples if training_samples else float("nan")
            validation_mean = validation_loss_sum / validation_samples if validation_samples else float("nan")
            loss_history["train"].append(train_mean)
            loss_history["validation"].append(validation_mean)
            if not math.isfinite(train_mean + validation_mean):
                raise RuntimeError("Non-finite loss; stop and inspect")
            history_path = experiment_dir / "results" / (run_name + "_history.json")
            history_tmp = history_path.with_suffix(".tmp")
            history_tmp.write_text(json.dumps(loss_history))
            history_tmp.replace(history_path)
            improved = validation_mean < best_validation_loss
            if improved:
                torch.save(autoencoder_1.state_dict(), experiment_dir / "checkpoint" / (run_name + "_minibatch_best.pt"))
            if (i + 1) % 25 == 0 or i == max_epoch - 1:
                torch.save({"epoch": i+1, "model": autoencoder_1.state_dict(), "optimizer": optimizer.state_dict(), "history": loss_history, "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state() if torch.cuda.is_available() else None, "loader_rng": train_loader.generator.get_state()}, experiment_dir / "checkpoint" / (run_name + "_resume.pt"))
            best_validation_loss = min(best_validation_loss, validation_mean)
            epoch_progress.set_postfix(train=f"{train_mean:.6f}", val=f"{validation_mean:.6f}", best=f"{best_validation_loss:.6f}")
            _tqdm.write(
                f"Epoch {i + 1:03d}/{max_epoch} | train {train_mean:.6f} | "
                f"validation {validation_mean:.6f} | {perf_counter() - started:.1f}s"
                + (" | New best validation" if improved else "")
            )
    
            fig, ax = plt.subplots(figsize=(9, 3.5))
            epochs = range(1, len(loss_history["train"]) + 1)
            ax.plot(epochs, loss_history["train"], label="Training", color="#6366f1", linewidth=2)
            ax.plot(epochs, loss_history["validation"], label="Validation (test split)", color="#f59e0b", linewidth=2)
            ax.set(xlabel="Epoch", ylabel="MSE loss", title="Autoencoder training progress")
            ax.grid(alpha=0.2)
            ax.legend()
            fig.tight_layout()
            fig.savefig(experiment_dir / "results" / (run_name + "_loss.png"), dpi=140)
            if plot_display is not None:
                plot_display.update(fig)
            plt.close(fig)
    
    # for epoch in range(num_epochs):
    #     for actions, frames, instructions, sample_indices in train_loader:
    #         # Training step
    #         pass
else:
    model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    autoencoder_1 = autoencoder_1.to(model_device)
    checkpoint_path = project_dir / "checkpoint" / (run_name + "_minibatch_" + CHECKPOINT_KIND + ".pt")
    autoencoder_1.load_state_dict(torch.load(checkpoint_path, map_location=model_device, weights_only=True))
    autoencoder_1.eval()
    print("Loaded", checkpoint_path.name, "on", model_device)


In [ ]:
if RUN_TRAINING:
    torch.save(autoencoder_1.state_dict(), experiment_dir / "checkpoint" / (run_name + "_minibatch_final.pt"))
    print("Completed 2000 epochs:", run_name)


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(experiment_dir / "results" / (run_name + "_loss.png"))))

In [ ]:
# All reconstruction results in one synchronized display.
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import FileLink, display
from pathlib import Path
import os

sample_indices_to_compare = (100, 200)
result_dataset = ball_moving_dataset_train  # Use ball_moving_dataset_test for held-out clips.
result_device = next(autoencoder_1.parameters()).device
result_videos = []
result_was_training = autoencoder_1.training
autoencoder_1.eval()
try:
    with torch.no_grad():
        for index in sample_indices_to_compare:
            _, frames, instruction, actual_index = result_dataset[index]
            inputs = frames.unsqueeze(0).permute(0, 4, 1, 2, 3).contiguous().float().to(result_device) / 255.0
            reconstruction = autoencoder_1.decode(autoencoder_1.encode(inputs))
            original_video = inputs[0].permute(1, 2, 3, 0).cpu()
            reconstructed_video = reconstruction[0].permute(1, 2, 3, 0).cpu()
            result_videos.append((original_video, reconstructed_video))
            print(f"Sample {actual_index} | MSE: {(original_video - reconstructed_video).square().mean().item():.6f}")
            print(instruction)
finally:
    autoencoder_1.train(result_was_training)

# Every panel uses the same frame index and a fixed RGB range of 0–1.
result_frame_count = min(len(original) for original, _ in result_videos)
result_panels = []
result_titles = []
for index, (original, reconstructed) in zip(sample_indices_to_compare, result_videos):
    result_panels.extend([original[:result_frame_count], reconstructed[:result_frame_count], (original - reconstructed).abs()[:result_frame_count]])
    result_titles.extend([f"Sample {index}\nOriginal", f"Sample {index}\nReconstruction", f"Sample {index}\nAbsolute error"])
result_panels.append((result_videos[0][1][:result_frame_count] - result_videos[1][1][:result_frame_count]).abs())
result_titles.append("Between reconstructions\nAbsolute difference")

result_figure, result_axes = plt.subplots(1, len(result_panels), figsize=(21, 3.5))
result_artists = []
for ax, video, title in zip(result_axes, result_panels, result_titles):
    result_artists.append(ax.imshow(video[0].clamp(0, 1).numpy()))
    ax.set_title(title, fontsize=10)
    ax.axis("off")
result_frame_title = result_figure.suptitle("Frame 0")
result_figure.tight_layout()

def update_result_frame(frame):
    for artist, video in zip(result_artists, result_panels):
        artist.set_data(video[frame].clamp(0, 1).numpy())
    result_frame_title.set_text(f"Frame {frame} / {result_frame_count - 1}")
    return [*result_artists, result_frame_title]

result_animation = FuncAnimation(result_figure, update_result_frame, frames=result_frame_count, interval=1000 / 30, blit=False)
# Keep the full animation outside the notebook so saving stays fast.
result_dir = experiment_dir / "results" if RUN_TRAINING else project_dir / "runs" / "reconstruction"
result_dir.mkdir(parents=True, exist_ok=True)
result_path = result_dir / "train_cnn_vae_3_level_minibatch.html"
try:
    with plt.rc_context({"animation.embed_limit": float("inf")}):
        result_path.write_text(result_animation.to_jshtml(fps=30), encoding="utf-8")
finally:
    plt.close(result_figure)
print(f"Saved video: {result_path}")
print("Open the saved HTML file in your browser to play.")
display(FileLink(os.path.relpath(result_path, Path.cwd())))
